# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [6]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [7]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/self-reliance.pdf")

docs = loader.load()

Let's look at an example document to see if everything worked as expected!

In [9]:
docs[0].page_content

'Self-Reliance\nRalph Waldo Emerson\n1841\n\\Ne te quaesiveris extra."\n\\Man is his own star; and the soul that can\nRender an honest and a perfect man,\nCommands all light, all in\ruence, all fate;\nNothing to him falls early or too late.\nOur acts our angels are, or good or ill,\nOur fatal shadows that walk by us still."\nEpilogue to Beaumont and Fletcher\'s Honest Man\'s Fortune\nCast the bantling on the rocks,\nSuckle him with the she-wolf\'s teat;\nWintered with the hawk and fox,\nPower and speed be hands and feet.\nI read the other day some verses written by an eminent painter which\nwere original and not conventional. The soul always hears an admonition\nin such lines, let the subject be what it may. The sentiment they instil is\nof more value than any thought they may contain. To believe your own\nthought, to believe that what is true for you in your private heart is true\nfor all men, | that is genius. Speak your latent conviction, and it shall\nbe the universal sense; for th

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [10]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    docs,
    embeddings,
    location=":memory:",
    collection_name="Ralph Waldo Emerson"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [11]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

In [12]:
# Size information about naive_retriever
print(f"Number of documents in the vector store: {len(docs)}")
print(f"Number of documents retrieved per query (k): {naive_retriever.search_kwargs['k']}")
print(f"Retriever type: {type(naive_retriever).__name__}")


Number of documents in the vector store: 21
Number of documents retrieved per query (k): 10
Retriever type: VectorStoreRetriever


### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [14]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [15]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [16]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to self-reliance and personal development, as the provided context is from Ralph Waldo Emerson\'s essay "Self-Reliance," which emphasizes individualism, inner strength, and independence.'

In [17]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific use cases or discussions about security. The text mainly focuses on themes of self-reliance, individuality, virtue, and society.'

In [18]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive opinions about the fintech projects. They recognized the importance of technological progress and innovation in finance, emphasizing the value of original and independent thinking. The judges appreciated projects that demonstrated genuine self-reliance, integrity, and a clear understanding of the principles underlying financial technology. They valued efforts that stood alone, without relying heavily on external support, and that contributed to the advancement of banking and finance through innovative solutions. Overall, the judges praised the fintech projects for their originality, impact, and the way they embody the spirit of self-trust and ingenuity.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [19]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)

We'll construct the same chain - only changing the retriever.

In [20]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [21]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided context appears to be related to philosophy, self-reliance, and individual thought. The text heavily emphasizes themes such as intuition, self-trust, originality, and the importance of individual perception and action. It discusses concepts of genius, virtue, and the divine within the individual, which suggests that the primary focus or domain is on personal development, philosophy, and moral self-reliance.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases or discussions specifically about security.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges had to express their opinions on the importance of self-reliance and nonconformity, emphasizing that society tends to favor conformity and the suppression of individual originality. They highlighted that true virtue and personal integrity come from being true to oneself, rather than succumbing to societal pressures or traditional standards. The judges recognized that society often works against the man's independence and that only by trusting and cultivating one's own mind can a person achieve genuine greatness and fulfillment."

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer 

For keyword lookups where lexical differnce is key


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [24]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [25]:
contextual_compression_retrieval_chain = (
    {
        "context": itemgetter("question") | compression_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))

    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided context, the document discusses themes related to individual self-reliance, personal virtue, intuition, and the importance of personal action over external authority or possessions. It emphasizes the value of private deeds, innate wisdom, and the individual's inner strength rather than external projects or domains.\n\nHowever, the specific most common project domain is not explicitly stated in the excerpt. Given the context's focus and content, it appears the overarching theme is self-development or personal growth.\n\nTherefore, I do not have enough information to determine a specific project domain from the provided context."

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'The provided context does not mention any specific use cases related to security.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had to say very little about the fintech projects, as the provided context does not include any information or statements from judges regarding these projects.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [29]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [30]:
multi_query_retrieval_chain = (
    {
        "context": itemgetter("question") | multi_query_retriever, 
        "question": itemgetter("question")
        }
    
    | RunnablePassthrough.assign(context=itemgetter("context"))

    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

In [31]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided context appears to be centered around the theme of self-reliance, individual virtue, and personal development. Ralph Waldo Emerson emphasizes the importance of independence, inner strength, and authentic action over conformity, societal expectations, and reliance on external institutions. The text explores ideas related to personal integrity, nonconformity, spontaneity, and living in accordance with one’s own nature.'

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'The provided context does not mention any specific use cases related to security.'

In [33]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had to say about the fintech projects is not mentioned in the provided context.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

When the question is reformualted, it could fetch slightly different embedded documents, all of which might be relevant. 

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [34]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = docs
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [35]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", 
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"), 
    client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [36]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [37]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [38]:
parent_document_retrieval_chain = (
    {
        "context": itemgetter("question") | parent_document_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, it appears that the most common project domain is not explicitly stated. The text discusses themes such as self-reliance, nature, virtue, the limitations of systems and classifications, the importance of staying at home rather than traveling, and individual power and integrity. It emphasizes philosophical ideas rather than specific project domains.\n\nTherefore, I do not know the most common project domain from the information given.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

"Based on the provided context, there are no explicit or detailed use cases about security mentioned. The text focuses more on themes of self-reliance, individualism, society's conformity, and the nature of change and progress, rather than security issues."

In [41]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges had positive and encouraging remarks about the fintech projects, emphasizing the importance of self-reliance, original thinking, and acting with integrity. They highlighted that true strength and success come from standing alone, trusting in one's principles, and not conforming to societal pressures. The judges appreciated projects that demonstrated innovation, independence, and the courage to pursue original ideas, reflecting core values of self-trust and authentic effort."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [43]:
ensemble_retrieval_chain = (
    {
        "context": itemgetter("question") | ensemble_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

Let's look at our results!

In [44]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be centered around self-reliance, individualism, and personal development. The content emphasizes themes such as trusting oneself, independence, nonconformity, virtue, and the importance of inner strength. These ideas suggest that the primary focus is on personal growth and philosophical reflection on how individuals can cultivate their own character and power.'

In [45]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no direct references or discussions about security use cases.'

In [46]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive remarks about the fintech projects. They recognized the projects as proof of the innovative and inventive spirit, describing them as an admirable form of invention and a good example of the power of human ingenuity. The projects were praised for their originality and for the demonstration of the creative force of the mind.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [48]:
semantic_documents = semantic_chunker.split_documents(docs[:20])

Let's create a new vector store.

In [49]:
# semantic_documents


In [50]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [51]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [52]:
semantic_retrieval_chain = (
    {
        "context": itemgetter("question") | semantic_retriever, 
        "question": itemgetter("question")
        }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {
        "response": rag_prompt | chat_model, 
        "context": itemgetter("context")
        }
)

And view the results!

In [53]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be related to individual self-reliance, personal virtue, and internal development. The text emphasizes themes such as self-trust, personal integrity, and the importance of individual action and character. There are references to historical figures, systems of thought, and the importance of personal effort over external systems or authorities.\n\nTherefore, the most common project domain is **personal development and self-reliance**.'

In [54]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific mentions or discussions about security use cases.'

In [55]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges' opinions about the fintech projects are not explicitly mentioned in the provided context."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

One way would be to make the percentile higher. The default is 95%, so maybe it could be set to like 98%. 
Gradient could also be used, which is specialized for highly repetative data. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### YOUR CODE HERE

1. generate dateset
    2. this will have cols like question, reference_context, reference
2. run the model on each question, get the actual context, and the answer
3. evaluate reference vs the answer with like helpfulnes and other metrics

In [56]:
from ragas.testset import TestsetGenerator

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_68677/3359797549.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_68677/3359797549.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [62]:
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/21 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/35 [00:00<?, ?it/s]

Property 'summary' already exists in node 'cb0d6c'. Skipping!
Property 'summary' already exists in node '3a389a'. Skipping!
Property 'summary' already exists in node '9a6955'. Skipping!
Property 'summary' already exists in node '8e0ab5'. Skipping!
Property 'summary' already exists in node '306e7c'. Skipping!
Property 'summary' already exists in node '97c80d'. Skipping!
Property 'summary' already exists in node '14c515'. Skipping!
Property 'summary' already exists in node '339173'. Skipping!
Property 'summary' already exists in node '0c7399'. Skipping!
Property 'summary' already exists in node 'bf4f50'. Skipping!
Property 'summary' already exists in node '29658e'. Skipping!
Property 'summary' already exists in node 'bf2f2a'. Skipping!
Property 'summary' already exists in node '9e01b8'. Skipping!
Property 'summary' already exists in node 'effda2'. Skipping!
Property 'summary' already exists in node '4ccd41'. Skipping!
Property 'summary' already exists in node '867904'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

...

Applying NERExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [63]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the concept of Jesus relate to the id...,[in this or that public place? Suppose you sho...,The context mentions that to be great is to be...,single_hop_specific_query_synthesizer
1,How does the metaphor of a hundred tacks relat...,[The voyage of the best ship is a zigzag line ...,The voyage of the best ship is described as a ...,single_hop_specific_query_synthesizer
2,Judas what mean in spirit?,"[When good is near you, when you have life in ...",The context discusses Judas as a figure who is...,single_hop_specific_query_synthesizer
3,How does the concept of America reflect the sp...,[Society never advances. It recedes as fast on...,The provided context discusses society's conti...,single_hop_specific_query_synthesizer
4,How does the voyage of a ship illustrate the i...,[<1-hop>\n\nin this or that public place? Supp...,"The voyage of the best ship, described as a zi...",multi_hop_abstract_query_synthesizer
5,How does misunderstanding relate to greatness ...,[<1-hop>\n\nin this or that public place? Supp...,The context suggests that being misunderstood ...,multi_hop_abstract_query_synthesizer
6,How does societal progress relate to the impac...,[<1-hop>\n\nSociety never advances. It recedes...,The context suggests that societal progress is...,multi_hop_abstract_query_synthesizer
7,How does the concept of Jesus relate to the id...,"[<1-hop>\n\nWhen good is near you, when you ha...",The context explores the nature of inner power...,multi_hop_specific_query_synthesizer
8,How does the divine example of Jesus relate to...,"[<1-hop>\n\nWhen good is near you, when you ha...",The context emphasizes that virtue is Height a...,multi_hop_specific_query_synthesizer
9,How does the concept of Jesus relate to the id...,"[<1-hop>\n\nWhen good is near you, when you ha...",The provided context discusses the nature of d...,multi_hop_specific_query_synthesizer


In [64]:
import time
from copy import deepcopy
from datetime import datetime

from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.metrics import (ContextEntityRecall,
                           LLMContextPrecisionWithoutReference,
                           LLMContextRecall, NoiseSensitivity,
                           ResponseRelevancy)

from langchain_community.callbacks import get_openai_callback

from ragas.cost import get_token_usage_for_openai


In [65]:
retrievers = {}

retrievers['baseline'] = naive_retrieval_chain
retrievers['bm25_retrieval_chain'] = bm25_retrieval_chain
retrievers['contextual_compression_retrieval_chain'] = contextual_compression_retrieval_chain
retrievers['multi_query_retrieval_chain'] = multi_query_retrieval_chain
retrievers['parent_document_retrieval_chain'] = parent_document_retrieval_chain
retrievers['ensemble_retrieval_chain'] = ensemble_retrieval_chain
retrievers['semantic_retrieval_chain'] = semantic_retrieval_chain



In [66]:
from collections import defaultdict

all_results = {}
num_tokens = defaultdict(list)
latencies = defaultdict(list)


for retreiver_chain_name, retreiver_chain in retrievers.items():
    print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print(f'testing {retreiver_chain_name}')
    
    dataset_for_this_retriever = deepcopy(dataset)

    x = 0
    for test_row in dataset_for_this_retriever:
        start_time = time.time()
        
        with get_openai_callback() as cb:
            response = retreiver_chain.invoke({"question" : test_row.eval_sample.user_input})
            num_tokens[retreiver_chain_name].append(cb.total_tokens)
        
        end_time = time.time()
        duration = end_time - start_time
        latencies[retreiver_chain_name].append(duration)
        
        test_row.eval_sample.response = response["response"].content
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        time.sleep(7) # To try to avoid rate limiting.
        print(x); x += 1


    evaluation_dataset = EvaluationDataset.from_pandas(dataset_for_this_retriever.to_pandas())

    print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print('running the evaluation now ...')
    evaluation_result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextPrecisionWithoutReference(), LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity(), ResponseRelevancy()],
        llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),
        run_config=RunConfig(timeout=360),
    )

    all_results[retreiver_chain_name] = evaluation_result

    print()
    print()



2025-10-12 15:07:15
testing baseline
0
1
2
3
4
5
6
7
8
9
10
2025-10-12 15:09:12
running the evaluation now ...


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_68677/3582310101.py:39: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),


Evaluating:   0%|          | 0/55 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[38]: TimeoutError()
Exception raised in Job[43]: TimeoutError()




2025-10-12 15:16:01
testing bm25_retrieval_chain
0
1
2
3
4
5
6
7
8
9
10
2025-10-12 15:17:49
running the evaluation now ...


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[33]: TimeoutError()
Exception raised in Job[38]: TimeoutError()
Exception raised in Job[43]: TimeoutError()
Exception raised in Job[48]: TimeoutError()
Exception raised in Job[53]: TimeoutError()




2025-10-12 15:54:07
testing semantic_retrieval_chain
0
1
2
3
4
5
6
7
8
9
10
2025-10-12 15:55:58
running the evaluation now ...


/var/folders/36/s6cm4m0959x3z451g1xvcyy40000gn/T/ipykernel_68677/3582310101.py:39: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini")),


Evaluating:   0%|          | 0/55 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[38]: TimeoutError()
Exception raised in Job[43]: TimeoutError()
Exception raised in Job[53]: TimeoutError()


In [73]:
all_results

{'baseline': {'llm_context_precision_without_reference': 0.9124, 'context_recall': 0.8500, 'context_entity_recall': 0.3334, 'noise_sensitivity(mode=relevant)': 0.2794, 'answer_relevancy': 0.9497},
 'bm25_retrieval_chain': {'llm_context_precision_without_reference': 0.9470, 'context_recall': 0.6636, 'context_entity_recall': 0.3555, 'noise_sensitivity(mode=relevant)': 0.2494, 'answer_relevancy': 0.8817},
 'contextual_compression_retrieval_chain': {'llm_context_precision_without_reference': 0.9091, 'context_recall': 0.7273, 'context_entity_recall': 0.3501, 'noise_sensitivity(mode=relevant)': 0.2034, 'answer_relevancy': 0.7723},
 'multi_query_retrieval_chain': {'llm_context_precision_without_reference': 0.8878, 'context_recall': 0.8455, 'context_entity_recall': 0.3501, 'noise_sensitivity(mode=relevant)': 0.0667, 'answer_relevancy': 0.8604},
 'parent_document_retrieval_chain': {'llm_context_precision_without_reference': 0.9545, 'context_recall': 0.6000, 'context_entity_recall': 0.3820, 'noi

In [74]:
import pandas

all_scores_as_df = pandas.DataFrame()

for retriever_name, evaluation_result in all_results.items():

    evalutation_result__all_scores = evaluation_result.scores
    evalutation_result__mean = pandas.DataFrame(evalutation_result__all_scores).mean()

    all_scores_as_df[retriever_name] = evalutation_result__mean

        
all_scores_as_df = all_scores_as_df.T


In [75]:
all_scores_as_df

,llm_context_precision_without_reference,context_recall,context_entity_recall,noise_sensitivity(mode=relevant),answer_relevancy
baseline,0.912396,0.850000,0.333442,0.279375,0.949718
bm25_retrieval_chain,0.946970,0.663636,0.355519,0.249368,0.881680
contextual_compression_retrieval_chain,0.909091,0.727273,0.350108,0.203382,0.772290
multi_query_retrieval_chain,0.887803,0.845455,0.350108,0.066667,0.860385
parent_document_retrieval_chain,0.954545,0.600000,0.382035,0.185027,0.872646
ensemble_retrieval_chain,0.889114,0.827273,0.353896,0.076923,0.873741
semantic_retrieval_chain,0.890933,0.754545,0.365981,0.318423,0.877309


* **LLMContextPrecisionWithoutReference** – checks if the answer stays within the context (no hallucinations). **Higher is better.**
* **LLMContextRecall** – checks if the answer uses the right parts of the context. **Higher is better.**
* **ContextEntityRecall** – checks if key names, facts, or entities from the context appear in the answer. **Higher is better.**
* **NoiseSensitivity** – checks how much irrelevant context throws off the answer. **Lower is better.**
* **ResponseRelevancy** – checks if the answer actually responds to the question. **Higher is better.**


all_scores_as_df

In [76]:
all_scores_as_df["noise_sensitivity(mode=relevant)"] = 1 - all_scores_as_df["noise_sensitivity(mode=relevant)"]


In [78]:
all_scores_as_df

,llm_context_precision_without_reference,context_recall,context_entity_recall,noise_sensitivity(mode=relevant),answer_relevancy
baseline,0.912396,0.850000,0.333442,0.720625,0.949718
bm25_retrieval_chain,0.946970,0.663636,0.355519,0.750632,0.881680
contextual_compression_retrieval_chain,0.909091,0.727273,0.350108,0.796618,0.772290
multi_query_retrieval_chain,0.887803,0.845455,0.350108,0.933333,0.860385
parent_document_retrieval_chain,0.954545,0.600000,0.382035,0.814973,0.872646
ensemble_retrieval_chain,0.889114,0.827273,0.353896,0.923077,0.873741
semantic_retrieval_chain,0.890933,0.754545,0.365981,0.681577,0.877309


In [94]:
import numpy as np

metric_cols                = all_scores_as_df.columns  # or explicitly list them
ranked_df                  = all_scores_as_df.rank(axis=0, method="min", ascending=False)
ranked_df["geo_mean_rank"] = np.exp(np.log(ranked_df[metric_cols]).mean(axis=1)).round(1)

# Add average latency for each retriever
avg_latencies = {name: np.mean(latencies[name]) for name in latencies.keys()}
ranked_df["avg_latency_seconds"] = ranked_df.index.map(avg_latencies).round(0)


avg_tokens = {name: np.mean(num_tokens[name]) for name in latencies.keys()}
ranked_df["avg_num_tokens"] = ranked_df.index.map(avg_tokens).round(0)

ranked_df


,llm_context_precision_without_reference,context_recall,context_entity_recall,noise_sensitivity(mode=relevant),answer_relevancy,geo_mean_rank,avg_latency_seconds,avg_num_tokens
baseline,3.0,1.0,7.0,6.0,1.0,2.6,4.0,8078.0
bm25_retrieval_chain,2.0,6.0,3.0,5.0,2.0,3.2,3.0,3295.0
contextual_compression_retrieval_chain,4.0,5.0,6.0,4.0,7.0,5.1,3.0,2668.0
multi_query_retrieval_chain,7.0,2.0,5.0,1.0,6.0,3.3,7.0,10285.0
parent_document_retrieval_chain,1.0,7.0,1.0,3.0,5.0,2.5,3.0,2654.0
ensemble_retrieval_chain,6.0,3.0,4.0,2.0,4.0,3.6,6.0,10784.0
semantic_retrieval_chain,5.0,4.0,2.0,7.0,3.0,3.8,3.0,4064.0
